# ICD-O Diagnosis Group Mapper: Development & Testing Notebook

This notebook serves as the development environment to create unit test and integration tests for the icdo diagnosis mapper tool.

In [1]:
import unittest
import pandas as pd
import subprocess
import os
from pathlib import Path

In [2]:
test_data = {'code':['8000/0', '8000/1', '8000/2','8000/3', '8000/6',  '8011/2'],
            'terms':["Neoplasm, benign","Neoplasm, uncertain whether benign or malignant",
                     "unknown","Neoplasm, malignant","Neoplasm, metastatic","unknown"]}
test_data = pd.DataFrame(test_data)
test_data

,code,terms
0,8000/0,"Neoplasm, benign"
1,8000/1,"Neoplasm, uncertain whether benign or malignant"
2,8000/2,unknown
3,8000/3,"Neoplasm, malignant"
4,8000/6,"Neoplasm, metastatic"
5,8011/2,unknown


In [3]:
expected_data = {
            'code': ['8000/0', '8000/1', '8000/2', '8000/3', '8000/6', '8011/2'],
            'term': [
                "Neoplasm, benign", 
                "Neoplasm, uncertain whether benign or malignant",
                "unknown", 
                "Neoplasm, malignant", 
                "Neoplasm, metastatic", 
                "unknown"
            ],
            'range': ['800', '800', '800', '800', '800', '801-804'],
            'group': [
                "Neoplasms, NOS", "Neoplasms, NOS", "Neoplasms, NOS", 
                "Neoplasms, NOS", "Neoplasms, NOS", "Epithelial neoplasms, NOS"
            ]
        }
expected_data = pd.DataFrame(test_data)
expected_data

,code,terms
0,8000/0,"Neoplasm, benign"
1,8000/1,"Neoplasm, uncertain whether benign or malignant"
2,8000/2,unknown
3,8000/3,"Neoplasm, malignant"
4,8000/6,"Neoplasm, metastatic"
5,8011/2,unknown


In [5]:
class TestScript(unittest.TestCase):
    def setUp(self):
        """Define paths and create test data."""
        # obtain and confirm folder and file paths
        self.root = Path(__file__).resolve().parent.parent
        self.input_dir = self.root / "input"
        self.input_dir.mkdir(exist_ok=True)

        self.output_dir = self.root / "output"
        self.output_dir.mkdir(exist_ok=True)
        self.output_files = []

        self.script_path = self.root / "scripts" / "icdo_group_mapper.py"

        # create a test input in the input folder
        test_data = {'code':['8000/0', '8000/1', '8000/2','8000/3', '8000/6',  '8011/2'],
                    'terms':["Neoplasm, benign","Neoplasm, uncertain whether benign or malignant",
                     "unknown","Neoplasm, malignant","Neoplasm, metastatic","unknown"]}
        test_data = pd.DataFrame(test_data)
        self.test_file = self.input_dir / "test_input.csv"
        pd.DataFrame(test_data).to_csv(self.test_file, index=False)

    def test_script_execution(self):
        """Run the script and check if it produces output."""
        # run the script
        result = subprocess.run(['python', str(self.script_path)], capture_output=True, text=True)
        
        # check script ran & output file was created
        self.assertEqual(result.returncode, 0, f"Script crashed with error: {result.stderr}")
        self.output_files = list(self.output_dir.glob("test_input_icdomapper_output_*.tsv"))
        self.assertTrue(len(self.output_files) > 0, "No output file was generated in /output")

    def tearDown(self):
        """clean up the test files."""
        if self.test_file.exists():
            os.remove(self.test_file)
        for out_file in self.output_files:
            if out_file.exists():
            os.remove(out_file)

#if __name__ == '__main__':
    #unittest.main()